# Hospitais Habilitados — Análise Exploratória de Dados

**Projeto:** TCC — Análise de Dados de Câncer de Mama em Municípios de Minas Gerais  
**Metodologia:** KDD (Knowledge Discovery in Databases)  
**Etapa:** Análise Exploratória de Dados (EDA)  
**Escopo:** Distribuição de hospitais habilitados em Oncologia, Reconstrução Mamária e Tratamento Sincrônico por município e regional de saúde de MG

---
## 1. Imports

In [ ]:
import json
import re
import unicodedata
import warnings
from pathlib import Path

import geobr
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

for renderer in ("plotly_mimetype", "vscode", "notebook_connected", "notebook"):
    if renderer in pio.renderers:
        pio.renderers.default = renderer
        break

---
## 2. Configurações Gerais

Centraliza todos os parâmetros configuráveis: caminhos, paleta de cores, ordem da legenda e constantes de layout.

In [ ]:
# ── Caminhos dos arquivos de entrada ─────────────────────────────────────────
CSV_HOSPITAIS  = Path("/home/anna/tcc/Tratamento/Hospitais/tabela_hospitais_habilitados.csv")
CSV_MUNICIPIOS = Path("/home/anna/tcc/Tratamento/Municípios/tabela_municipios_regionais.csv")
ANO_MALHA_MG   = 2020

# ── Mapeamento de tipos (normalizado → display) ────────────────────────────────
MAPA_TIPOS = {
    "oncologia"            : "Oncologia",
    "reconstrucao mamaria" : "Reconstrução Mamária",
    "tratamento sincronico": "Tratamento Sincrônico",
}

# ── Paleta de cores por combinação de tipos ───────────────────────────────────
CORES_TIPOS = {
    "Reconstrução Mamária"                                    : "#74a8db",
    "Reconstrução Mamária e Oncologia"                        : "#ff7f50",
    "Reconstrução Mamária e Tratamento Sincrônico"            : "#8e1e1e",
    "Reconstrução Mamária, Tratamento Sincrônico, Oncologia"  : "#9c2328",
    "Tratamento Sincrônico"                                   : "#3e78b3",
    "Oncologia"                                               : "#fec44f",
    "Nenhum"                                                  : "#ffffd4",
}

# ── Ordem da legenda nos mapas ────────────────────────────────────────────────
ORDEM_LEGENDA = [
    "Reconstrução Mamária",
    "Reconstrução Mamária e Oncologia",
    "Reconstrução Mamária e Tratamento Sincrônico",
    "Reconstrução Mamária, Tratamento Sincrônico, Oncologia",
    "Tratamento Sincrônico",
    "Oncologia",
    "Nenhum",
]

# ── Layout dos mapas ──────────────────────────────────────────────────────────
ZOOM_MAPA   = 5.5
ALTURA_MAPA = 760
OPACIDADE   = 0.95

# ── Layout das tabelas e gráficos de ranking ──────────────────────────────────
TOP_N             = 15          # número de itens no ranking
LARGURA_GRAFICO   = 1000
ALTURA_RANKING    = 550
FONTE_TITULO      = 20
FONTE_EIXO_TITULO = 15
FONTE_EIXO_TICK   = 13
FONTE_TEXTO_BARRA = 13
COR_BARRA_MUN     = "#2D9CDB"
COR_BARRA_REG     = "#D7263D"

---
## 3. Funções Auxiliares

In [ ]:
def normalizar_texto(valor) -> str:
    """Remove acentos e converte para minúsculas. Trata NaN como string vazia."""
    if pd.isna(valor):
        return ""
    txt = str(valor).strip().lower()
    txt = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode("utf-8")
    return re.sub(r"\s+", " ", txt)


def combinar_tipos(series_tipos) -> str:
    """
    Classifica um conjunto de tipos de habilitação em uma categoria combinada.
    Usado para colorir o mapa de acordo com a combinação presente no município/regional.
    """
    tipos = {t for t in series_tipos if pd.notna(t)}
    combos = {
        frozenset(["Reconstrução Mamária"])                                    : "Reconstrução Mamária",
        frozenset(["Reconstrução Mamária", "Oncologia"])                       : "Reconstrução Mamária e Oncologia",
        frozenset(["Reconstrução Mamária", "Tratamento Sincrônico"])           : "Reconstrução Mamária e Tratamento Sincrônico",
        frozenset(["Reconstrução Mamária", "Tratamento Sincrônico","Oncologia"]): "Reconstrução Mamária, Tratamento Sincrônico, Oncologia",
        frozenset(["Tratamento Sincrônico"])                                   : "Tratamento Sincrônico",
        frozenset(["Oncologia"])                                               : "Oncologia",
    }
    return combos.get(frozenset(tipos), "Nenhum")


def _centro_gdf(gdf) -> tuple:
    """Retorna (lat_centro, lon_centro) do bounding box de um GeoDataFrame."""
    b = gdf.total_bounds
    return (b[1] + b[3]) / 2, (b[0] + b[2]) / 2


print("✅ Funções auxiliares definidas.")

---
## 4. Carregamento e Preparação dos Dados

Leitura dos arquivos de hospitais habilitados e da tabela de municípios × regionais,
seguida de enriquecimento com regional de saúde e classificação do tipo combinado.

In [ ]:
for path in (CSV_HOSPITAIS, CSV_MUNICIPIOS):
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")

df_hosp    = pd.read_csv(CSV_HOSPITAIS)
df_mun_reg = pd.read_csv(CSV_MUNICIPIOS)
muni_mg    = geobr.read_municipality(code_muni="MG", year=ANO_MALHA_MG, simplified=True).to_crs(4326)

# ── Padronização de colunas ───────────────────────────────────────────────────
df_hosp.columns    = [c.strip() for c in df_hosp.columns]
df_mun_reg.columns = [c.strip() for c in df_mun_reg.columns]

# ── Normalização do DataFrame de hospitais ────────────────────────────────────
df_hosp["CODIGO_IBGE"]       = pd.to_numeric(df_hosp["CÓDIGO"], errors="coerce").astype("Int64")
df_hosp["MUNICIPIO_NORM"]    = df_hosp["MUNICÍPIOS"].map(normalizar_texto)
df_hosp["TIPO_CLASSIFICADO"] = df_hosp["TIPO"].map(normalizar_texto).map(MAPA_TIPOS)
df_hosp = df_hosp[df_hosp["TIPO_CLASSIFICADO"].notna()].copy()

# ── Normalização da tabela de municípios ──────────────────────────────────────
df_mun_reg["codigo_ibge"]           = pd.to_numeric(df_mun_reg["codigo_ibge"], errors="coerce").astype("Int64")
df_mun_reg["municipio_normalizado"] = df_mun_reg["municipio_normalizado"].fillna("").map(normalizar_texto)
df_mun_reg["regional_original"]     = df_mun_reg["regional_original"].fillna("Nenhum")

# Mapa nome_normalizado → regional (fallback quando código IBGE não bate)
mapa_regional_por_nome = (
    df_mun_reg[["municipio_normalizado", "regional_original"]]
    .drop_duplicates(subset=["municipio_normalizado"])
    .set_index("municipio_normalizado")["regional_original"]
    .to_dict()
)

# ── Join hospitais × regionais ────────────────────────────────────────────────
base_hosp = df_hosp.merge(
    df_mun_reg[["codigo_ibge", "regional_original"]],
    left_on="CODIGO_IBGE", right_on="codigo_ibge", how="left",
)
# Fallback por nome normalizado para hospitais sem código IBGE mapeado
mask_sem = base_hosp["regional_original"].isna() | base_hosp["regional_original"].eq("")
base_hosp.loc[mask_sem, "regional_original"] = (
    base_hosp.loc[mask_sem, "MUNICIPIO_NORM"].map(mapa_regional_por_nome)
)
base_hosp["regional_original"] = base_hosp["regional_original"].fillna("Nenhum")

print("╔══════════════════════════════════════════════════╗")
print("║           Dados carregados                       ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Hospitais habilitados : {len(df_hosp):<23} ║")
print(f"║  Municípios na malha   : {len(muni_mg):<23} ║")
print(f"║  Com regional (join)   : {int(base_hosp['regional_original'].ne('Nenhum').sum()):<23} ║")
print("╚══════════════════════════════════════════════════╝")

---
## 5. Análise Inicial dos Dados

Inspeção rápida do DataFrame de hospitais: distribuição por tipo e municípios cobertos.

In [ ]:
display(Markdown(f"**Shape:** {df_hosp.shape[0]} registros × {df_hosp.shape[1]} colunas"))

display(Markdown("**Distribuição por tipo de habilitação:**"))
display(
    df_hosp["TIPO_CLASSIFICADO"].value_counts().rename_axis("Tipo").reset_index(name="Qtd")
    .style
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
    .hide(axis="index")
)

display(Markdown(f"**Municípios únicos com ao menos um hospital habilitado:** "
                 f"{df_hosp['CODIGO_IBGE'].nunique()}"))

missing = df_hosp.isnull().sum()
missing = missing[missing > 0]
if not missing.empty:
    display(Markdown("**Valores ausentes por coluna:**"))
    display(missing.to_frame("Nulos").style.background_gradient(cmap="Reds"))
else:
    display(Markdown("✅ **Sem valores ausentes nas colunas principais.**"))

---
## 6. Preparação dos GeoDataFrames para Mapas

Agrega os hospitais por município e por regional de saúde, classificando a combinação de tipos.

In [ ]:
# ── Agregação por município ───────────────────────────────────────────────────
combo_municipio = (
    base_hosp.groupby("CODIGO_IBGE", as_index=False)
    .agg(
        categoria_mapa         =("TIPO_CLASSIFICADO", combinar_tipos),
        tipos_presentes        =("TIPO_CLASSIFICADO", lambda s: ", ".join(sorted(set(s)))),
        estabelecimentos_unicos=("ESTABELECIMENTO",   "nunique"),
        registros              =("ESTABELECIMENTO",   "count"),
    )
)

gdf_municipios = muni_mg.merge(combo_municipio, left_on="code_muni", right_on="CODIGO_IBGE", how="left")
for col, default in [("categoria_mapa","Nenhum"), ("tipos_presentes","Nenhum"),
                     ("estabelecimentos_unicos",0), ("registros",0)]:
    gdf_municipios[col] = gdf_municipios[col].fillna(default)
gdf_municipios["estabelecimentos_unicos"] = gdf_municipios["estabelecimentos_unicos"].astype(int)
gdf_municipios["registros"]               = gdf_municipios["registros"].astype(int)
gdf_municipios["categoria_mapa"] = pd.Categorical(
    gdf_municipios["categoria_mapa"], categories=ORDEM_LEGENDA, ordered=True
)
gdf_municipios = gdf_municipios.reset_index(drop=True)
gdf_municipios["id"] = gdf_municipios.index.astype(str)

# ── Agregação por regional ────────────────────────────────────────────────────
combo_regional = (
    base_hosp[base_hosp["regional_original"] != "Nenhum"]
    .groupby("regional_original", as_index=False)
    .agg(
        categoria_mapa         =("TIPO_CLASSIFICADO", combinar_tipos),
        tipos_presentes        =("TIPO_CLASSIFICADO", lambda s: ", ".join(sorted(set(s)))),
        estabelecimentos_unicos=("ESTABELECIMENTO",   "nunique"),
        registros              =("ESTABELECIMENTO",   "count"),
    )
)

gdf_mg_reg = muni_mg.merge(
    df_mun_reg[["codigo_ibge","regional_original"]],
    left_on="code_muni", right_on="codigo_ibge", how="left",
)
gdf_mg_reg["regional_original"] = gdf_mg_reg["regional_original"].fillna("Nenhum")

gdf_regionais = (
    gdf_mg_reg[gdf_mg_reg["regional_original"] != "Nenhum"][["regional_original","geometry"]]
    .dissolve(by="regional_original", as_index=False)
    .merge(combo_regional, on="regional_original", how="left")
)
for col, default in [("categoria_mapa","Nenhum"), ("tipos_presentes","Nenhum"),
                     ("estabelecimentos_unicos",0), ("registros",0)]:
    gdf_regionais[col] = gdf_regionais[col].fillna(default)
gdf_regionais["estabelecimentos_unicos"] = gdf_regionais["estabelecimentos_unicos"].astype(int)
gdf_regionais["registros"]               = gdf_regionais["registros"].astype(int)
gdf_regionais["categoria_mapa"] = pd.Categorical(
    gdf_regionais["categoria_mapa"], categories=ORDEM_LEGENDA, ordered=True
)
gdf_regionais = gdf_regionais.reset_index(drop=True)
gdf_regionais["id"] = gdf_regionais.index.astype(str)

print("✅ GeoDataFrames preparados.")
print(f"   Municípios com hospital habilitado: {int((gdf_municipios['categoria_mapa'] != 'Nenhum').sum())}")
print(f"   Regionais com hospital habilitado : {int((gdf_regionais['categoria_mapa'] != 'Nenhum').sum())}")

---
## 7. Mapas de Distribuição

### 7.1 Mapa por Município

In [ ]:
lat_c, lon_c = _centro_gdf(gdf_municipios)

fig_mun = px.choropleth_map(
    gdf_municipios,
    geojson=json.loads(gdf_municipios.to_json()),
    locations="id",
    color="categoria_mapa",
    color_discrete_map=CORES_TIPOS,
    category_orders={"categoria_mapa": ORDEM_LEGENDA},
    title="Distribuição de Hospitais Habilitados por Município em Minas Gerais",
    hover_name="name_muni",
    hover_data={
        "tipos_presentes"        : True,
        "estabelecimentos_unicos": True,
        "registros"              : True,
        "id"                     : False,
    },
    map_style="white-bg",
    center={"lat": lat_c, "lon": lon_c},
    zoom=ZOOM_MAPA,
    opacity=OPACIDADE,
)
fig_mun.update_layout(
    margin=dict(l=10, r=10, t=60, b=10),
    title_x=0.5,
    height=ALTURA_MAPA,
    legend_title_text="Combinação",
)
fig_mun.show()

### 7.2 Mapa por Regional de Saúde

In [ ]:
lat_c, lon_c = _centro_gdf(gdf_regionais)

fig_reg = px.choropleth_map(
    gdf_regionais,
    geojson=json.loads(gdf_regionais.to_json()),
    locations="id",
    color="categoria_mapa",
    color_discrete_map=CORES_TIPOS,
    category_orders={"categoria_mapa": ORDEM_LEGENDA},
    title="Distribuição de Hospitais Habilitados por Regional de Saúde em Minas Gerais",
    hover_name="regional_original",
    hover_data={
        "tipos_presentes"        : True,
        "estabelecimentos_unicos": True,
        "registros"              : True,
        "id"                     : False,
    },
    map_style="white-bg",
    center={"lat": lat_c, "lon": lon_c},
    zoom=ZOOM_MAPA,
    opacity=OPACIDADE,
)
fig_reg.update_layout(
    margin=dict(l=10, r=10, t=60, b=10),
    title_x=0.5,
    height=ALTURA_MAPA,
    legend_title_text="Combinação",
)
fig_reg.show()

---
## 8. Rankings

### 8.1 Top municípios por número de estabelecimentos habilitados

In [ ]:
ranking_mun = (
    base_hosp.groupby("MUNICIPIO_NORM", as_index=False)
    .agg(hospitais=("ESTABELECIMENTO", "nunique"))
    .sort_values("hospitais", ascending=False)
    .head(TOP_N)
    .sort_values("hospitais")   # crescente para gráfico horizontal
)

fig_rank_mun = go.Figure(go.Bar(
    x=ranking_mun["hospitais"],
    y=ranking_mun["MUNICIPIO_NORM"],
    orientation="h",
    marker_color=COR_BARRA_MUN,
    text=ranking_mun["hospitais"],
    textposition="outside",
    textfont=dict(size=FONTE_TEXTO_BARRA),
    cliponaxis=False,
    constraintext="none",
))
fig_rank_mun.update_layout(
    title=dict(text=f"Top {TOP_N} municípios por estabelecimentos habilitados",
               font=dict(size=FONTE_TITULO), x=0.5),
    xaxis=dict(title="Nº de estabelecimentos", title_font=dict(size=FONTE_EIXO_TITULO),
               tickfont=dict(size=FONTE_EIXO_TICK),
               range=[0, int(ranking_mun["hospitais"].max() * 1.2)],
               showgrid=True, gridcolor="#E5ECF6"),
    yaxis=dict(title="Município", title_font=dict(size=FONTE_EIXO_TITULO),
               tickfont=dict(size=FONTE_EIXO_TICK), automargin=True),
    template="plotly_white",
    height=ALTURA_RANKING, width=LARGURA_GRAFICO,
    margin=dict(l=20, r=80, t=70, b=20),
)
fig_rank_mun.show()

### 8.2 Top regionais por número de estabelecimentos habilitados

In [ ]:
ranking_reg = (
    base_hosp[base_hosp["regional_original"] != "Nenhum"]
    .groupby("regional_original", as_index=False)
    .agg(hospitais=("ESTABELECIMENTO", "nunique"))
    .sort_values("hospitais", ascending=False)
    .head(TOP_N)
    .sort_values("hospitais")
)

fig_rank_reg = go.Figure(go.Bar(
    x=ranking_reg["hospitais"],
    y=ranking_reg["regional_original"],
    orientation="h",
    marker_color=COR_BARRA_REG,
    text=ranking_reg["hospitais"],
    textposition="outside",
    textfont=dict(size=FONTE_TEXTO_BARRA),
    cliponaxis=False,
    constraintext="none",
))
fig_rank_reg.update_layout(
    title=dict(text=f"Top {TOP_N} regionais por estabelecimentos habilitados",
               font=dict(size=FONTE_TITULO), x=0.5),
    xaxis=dict(title="Nº de estabelecimentos", title_font=dict(size=FONTE_EIXO_TITULO),
               tickfont=dict(size=FONTE_EIXO_TICK),
               range=[0, int(ranking_reg["hospitais"].max() * 1.2)],
               showgrid=True, gridcolor="#E5ECF6"),
    yaxis=dict(title="Regional de Saúde", title_font=dict(size=FONTE_EIXO_TITULO),
               tickfont=dict(size=FONTE_EIXO_TICK), automargin=True),
    template="plotly_white",
    height=ALTURA_RANKING, width=LARGURA_GRAFICO,
    margin=dict(l=20, r=80, t=70, b=20),
)
fig_rank_reg.show()

---
## 9. Tabelas Detalhadas

### 9.1 Estabelecimentos por município (tabela completa)

In [ ]:
tabela_mun = (
    base_hosp.groupby("MUNICIPIO_NORM", as_index=False)
    .agg(
        estabelecimentos=("ESTABELECIMENTO", "nunique"),
        tipos           =("TIPO_CLASSIFICADO", lambda s: ", ".join(sorted(set(s)))),
    )
    .sort_values("estabelecimentos", ascending=False)
    .rename(columns={
        "MUNICIPIO_NORM"  : "Município",
        "estabelecimentos": "Estabelecimentos",
        "tipos"           : "Tipos de habilitação",
    })
    .reset_index(drop=True)
)

display(Markdown(f"**Total de municípios com hospital habilitado:** {len(tabela_mun)}"))
display(
    tabela_mun.style
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
    .hide(axis="index")
)

### 9.2 Estabelecimentos por município com lista detalhada

In [ ]:
tabela_lista = (
    base_hosp.groupby("MUNICIPIO_NORM")["ESTABELECIMENTO"]
    .apply(lambda x: sorted(set(x.tolist())))
    .reset_index()
    .rename(columns={"MUNICIPIO_NORM": "Município", "ESTABELECIMENTO": "Estabelecimentos"})
)
tabela_lista["Qtd"] = tabela_lista["Estabelecimentos"].apply(len)
tabela_lista = tabela_lista.sort_values("Qtd", ascending=False).reset_index(drop=True)

for _, row in tabela_lista.iterrows():
    qtd = row["Qtd"]
    print(f"\n{row['Município']} ({qtd} hospital{'is' if qtd > 1 else ''})")
    for h in row["Estabelecimentos"]:
        print(f"   - {h}")

### 9.3 Tabela por Regional de Saúde

In [ ]:
tabela_reg = (
    base_hosp[base_hosp["regional_original"] != "Nenhum"]
    .groupby("regional_original", as_index=False)
    .agg(
        estabelecimentos=("ESTABELECIMENTO", "nunique"),
        tipos           =("TIPO_CLASSIFICADO", lambda s: ", ".join(sorted(set(s)))),
    )
    .sort_values("estabelecimentos", ascending=False)
    .rename(columns={
        "regional_original": "Regional de Saúde",
        "estabelecimentos" : "Estabelecimentos",
        "tipos"            : "Tipos de habilitação",
    })
    .reset_index(drop=True)
)

display(Markdown(f"**Total de regionais com hospital habilitado:** {len(tabela_reg)}"))
display(
    tabela_reg.style
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
    .hide(axis="index")
)

---
## 10. Conclusões da EDA — Hospitais Habilitados

### Distribuição geográfica
- A maior concentração de hospitais habilitados está na região metropolitana de Belo Horizonte e nos municípios-sede das regionais de saúde.
- A maioria dos municípios de MG não possui nenhum hospital habilitado, evidenciando concentração geográfica da oferta.

### Tipos de habilitação
- **Oncologia** é o tipo mais frequente isoladamente.
- **Reconstrução Mamária** ocorre predominantemente em combinação com outros tipos.
- A combinação mais completa (Oncologia + Reconstrução Mamária + Tratamento Sincrônico) aparece em poucos municípios de grande porte.

### Qualidade dos dados
- Municípios sem código IBGE foram resolvidos via nome normalizado (fallback).
- O campo `TIPO` foi mapeado via `MAPA_TIPOS` para garantir padronização.

> **Próxima etapa (PFC II):** Cruzamento da disponibilidade de hospitais com indicadores de mortalidade e incidência por município para análise de correlação e clustering K-means.